# HW2 – 3. One-stage anchor-based detector (RetinaNet)

RetinaNet (ResNet-50 FPN v2, COCO-pretrained) trained under the fixed protocol from notebook 01 the same as Faster R-CNN in notebook 02 (same data, metrics, epochs and about the same training GPU time).

**Experiments** (each starts from the best configuration so far):
1. **Anchor sizes** – torchvision default vs anchors grounded in the object sizes from notebook 01.
2. **IoU thresholds of the anchor assignment** – positive/negative 0.5/0.4 (default) vs 0.4/0.3.
3. *(optional, if time allows)* **Box regression loss** – GIoU (default) vs smooth L1.

The tuned model is the best training with the better final NMS IoU. We compare it with the tuned two-stage model from notebook 02, discuss what removing the proposal stage costs and analyse failure cases.

## 0. Setup

In [ ]:
# Make the repo importable: Kaggle clones it, locally the root is the parent of notebooks/
import os, sys
from pathlib import Path
if os.path.exists("/kaggle/working"):
    if not os.path.exists("/kaggle/working/repo"):
        !git clone --depth 1 https://github.com/BohdanZasymovych/ai-hw2-object-detection.git /kaggle/working/repo
    !pip install -q pycocotools huggingface_hub   # only if missing; torch/torchvision stay Kaggle's own CUDA builds
    ROOT_DIR = Path("/kaggle/working/repo")
else:
    ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

In [ ]:
# Live reload of utils/common.py when editing locally (Kaggle's IPython autoreload is broken on Python 3.13)
if not os.path.exists("/kaggle/working"):
    %load_ext autoreload
    %autoreload 2
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torchvision.models.detection import retinanet_resnet50_fpn_v2, RetinaNet_ResNet50_FPN_V2_Weights
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.image_list import ImageList
from torchvision.models.detection._utils import Matcher
from torchvision.ops import box_iou
from utils.common import *

seed_everything()
print(env_info(), "| data:", RAW_DIR)

Stages run in this session are listed in `RUN`; finished stages are downloaded from the Hugging Face repo. `loss` is optional: the tuned model uses it only if it exists. `QUICK=True` is a local smoke test on a few frames.

In [ ]:
RUN = set()  # {"baseline", "anchors", "iou", "loss", "final"} stages computed in this session; the rest are pulled from the HF repo
QUICK = False                                    # True: local smoke test (few frames, 1 epoch, small input, nothing pushed)
RUN_SPEED_TEST = False

# Fail early if trained stages could not be uploaded (needs HF_TOKEN with write access)
if RUN and not QUICK:
    check_hf_write()

## 1. Training setup

- **Model:** RetinaNet ResNet-50 FPN v2 (torchvision), COCO weights. FPN levels P3–P7 (strides 8–128), 9 anchors per location (3 scales × aspect ratios 0.5/1/2), GroupNorm heads, focal loss (α 0.25, γ 2), GIoU box loss. Only the last classification layer is new (8 classes, bias prior 0.01); the rest of the head stays pretrained. Trainable backbone stages: torchvision default (3).
- **Data:** `PREP` protocol – every 3rd frame (8,090 train / 951 val), 1280 px long side, flip + scale jitter + colour jitter.
- **Optimisation:** SGD (momentum 0.9, weight decay 1e-4), lr 0.0025 (torchvision's RetinaNet recipe, 0.01 for batch 16, scaled linearly to batch 4 as in notebook 02), ×0.1 after epoch 2, 3 epochs, AMP, gradient clipping 10. Same seed and budget as notebook 02 (one run takes about 1 h on a T4, as Faster R-CNN).
- **Evaluation:** COCO metrics on val after every epoch (best epoch kept), ignore regions as `iscrowd`, up to 300 detections, score ≥ 0.001; speed at batch size 1 with AMP.

In [ ]:
CFG = dict(
    epochs=1 if QUICK else 3, batch_size=2 if QUICK else 4, lr=0.0025, momentum=0.9, weight_decay=1e-4, lr_drop_epoch=2,
    long_side=640 if QUICK else PREP["long_side"], train_frames=16 if QUICK else None, val_frames=8 if QUICK else None,
    num_workers=4, time_budget_min=110, seed=SEED,
)
DEFAULT_BASES = (32, 64, 128, 256, 512)   # torchvision default: base anchor size per level P3-P7, each x 2^0, 2^(1/3), 2^(2/3)
DATA_BASES = (13, 26, 52, 104, 208)       # smallest k-means size (notebook 01, section 2.5), doubled per level like the stride
DEFAULT_IOU = (0.5, 0.4)                  # anchor is positive if IoU >= 0.5 with a GT box, negative if < 0.4, ignored in between
LOW_IOU = (0.4, 0.3)
DEFAULT_MODEL = dict(bases=DEFAULT_BASES, iou=DEFAULT_IOU, box_loss="giou")
DEFAULT_TEST = dict(nms=0.5)              # torchvision test-time default
NMS = [0.5, 0.6]                          # final NMS IoU checked on the tuned model (test time, no retraining)


def name(stage):
    return f"retina_{stage}" + ("_quick" if QUICK else "")


def anchor_sizes(bases):
    # 3 scales per level as in torchvision's default anchor generator
    return tuple((b, int(b * 2 ** (1 / 3)), int(b * 2 ** (2 / 3))) for b in bases)


print(CFG)
print("default anchors:", anchor_sizes(DEFAULT_BASES), "\ndata anchors:   ", anchor_sizes(DATA_BASES))

## 2. Model and helpers

In [ ]:
def build_retina(bases=DEFAULT_BASES, iou=DEFAULT_IOU, box_loss="giou"):
    # COCO-pretrained RetinaNet R50-FPN v2 with an 8-class classifier, our input size, evaluation limits,
    # given anchor sizes, IoU thresholds of the anchor assignment and box regression loss
    m = retinanet_resnet50_fpn_v2(weights=RetinaNet_ResNet50_FPN_V2_Weights.COCO_V1, score_thresh=0.001,
                                  detections_per_img=EVAL_DETS, fg_iou_thresh=iou[0], bg_iou_thresh=iou[1],
                                  **tv_size_kwargs(CFG["long_side"]))
    # still 9 anchors per location, so the pretrained head fits
    m.anchor_generator = AnchorGenerator(anchor_sizes(bases), ((0.5, 1.0, 2.0),) * len(bases))
    cls = m.head.classification_head                              # new output layer, initialised as in torchvision
    cls.cls_logits = torch.nn.Conv2d(cls.cls_logits.in_channels, cls.num_anchors * len(CLASSES), 3, padding=1)
    torch.nn.init.normal_(cls.cls_logits.weight, std=0.01)
    torch.nn.init.constant_(cls.cls_logits.bias, -math.log((1 - 0.01) / 0.01))   # prior probability 0.01
    cls.num_classes = len(CLASSES)                                # labels 0..7 (no background class: sigmoid outputs)
    m.head.regression_head._loss_type = box_loss
    return m


def set_test_params(m, nms):
    # IoU of the final per-class NMS
    m.nms_thresh = nms
    return m


def ensure_run(stage, weights=False):
    # Local folder of a finished stage; downloaded from the HF repo if it was produced in another session
    d = RUNS_DIR / name(stage)
    if not (d / "metrics.json").exists() or (weights and not (d / "best.pt").exists()):
        pull_run(name(stage), weights=weights)
    if not (d / "metrics.json").exists():
        raise FileNotFoundError(f"Stage '{stage}' has no results locally or in the HF repo {HF_REPO}: "
                                f"add '{stage}' to RUN to compute it in this session (RUN = {RUN}).")
    return d


def model_cfg(stage):
    return json.load(open(ensure_run(stage) / "config.json"))["model"]


def load_model(stage):
    m = build_retina(**model_cfg(stage))
    m.load_state_dict(torch.load(ensure_run(stage, weights=True) / "best.pt", map_location="cpu"))
    return m.to(DEVICE).eval()


def evaluate(model, **kw):
    return evaluate_model(torchvision_predictor(model), "val", max_frames=CFG["val_frames"], batch_size=1,
                          num_workers=CFG["num_workers"], label_offset=0, **kw)


def best_so_far(stages):
    # Finished stage with the highest val mAP (optional stages that were never run are skipped)
    done = {}
    for s in stages:
        try:
            done[s] = json.load(open(ensure_run(s) / "metrics.json"))["mAP"]
        except FileNotFoundError:
            pass
    return max(done, key=done.get)

In [ ]:
def train_stage(stage, **model):
    # Train one configuration under the fixed protocol, keep the best epoch, save and upload the run
    seed_everything()
    d = run_dir(name(stage))
    net = build_retina(**model).to(DEVICE)
    opt = torch.optim.SGD([p for p in net.parameters() if p.requires_grad], lr=CFG["lr"],
                          momentum=CFG["momentum"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, [CFG["lr_drop_epoch"]], 0.1)
    loader = make_loader(DetDataset("train", train=True, label_offset=0, max_frames=CFG["train_frames"]), CFG["batch_size"],
                         num_workers=CFG["num_workers"])
    history = fit(net, loader, opt, CFG["epochs"], scheduler=sched, eval_fn=lambda: evaluate(net),
                  ckpt_path=d / "best.pt", last_path=d / "last.pt", resume_from=d / "last.pt",
                  time_budget_s=CFG["time_budget_min"] * 60)
    net.load_state_dict(torch.load(d / "best.pt", map_location=DEVICE))
    metrics = evaluate(net)
    save_run(name(stage), config={**CFG, "model": model, "test": DEFAULT_TEST}, history=history, metrics=metrics)
    if not QUICK:
        push_run(name(stage))
    return metrics

In [ ]:
@torch.inference_mode()
def assignment_stats(bases, iou, n_frames=10 if QUICK else 200):
    # How the anchors of one configuration are labelled during training, on train frames at the input resolution:
    # anchors / positives / negatives per image, positive anchors per GT box with IoU >= the threshold, and the share of GT
    # boxes that reach the threshold with no anchor (they only get their best anchor(s) as low-quality matches)
    gen, matcher = AnchorGenerator(anchor_sizes(bases), ((0.5, 1.0, 2.0),) * len(bases)), Matcher(*iou, allow_low_quality_matches=True)
    ds = DetDataset("train", max_frames=n_frames)
    cache, rows, per_img = {}, [], []
    for k in range(len(ds)):
        im, t = ds[k]
        b = t["boxes"]
        if not len(b):
            continue
        h, w = im.shape[1:]
        if (h, w) not in cache:                                  # anchors depend only on the (padded) image size
            H, W = -(-h // 32) * 32, -(-w // 32) * 32             # padding of model.transform (size_divisible=32)
            shapes = [(H // s, W // s) for s in (8, 16, 32)]      # P3-P5
            for _ in range(2):                                    # P6, P7: stride-2 convs
                shapes.append((-(-shapes[-1][0] // 2), -(-shapes[-1][1] // 2)))
            cache[(h, w)] = gen(ImageList(torch.zeros(1, 3, H, W), [(h, w)]), [torch.zeros(1, 1, *s) for s in shapes])[0].to(DEVICE)
        a = cache[(h, w)]
        q = box_iou(b.to(DEVICE), a)
        idx = matcher(q)
        pos = idx >= 0                                            # positives incl. low-quality matches (all ties of a GT's best IoU)
        above = q[idx[pos], pos.nonzero()[:, 0]] >= iou[0]         # ... of which really reach the threshold
        rows.append(pd.DataFrame({"size": ((b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])).sqrt().numpy(),
                                  "positives": torch.bincount(idx[pos][above], minlength=len(b)).cpu().numpy(),
                                  "low_quality_only": (q.max(1).values < iou[0]).cpu().numpy()}))
        per_img.append((len(a), (idx >= 0).sum().item(), (idx == Matcher.BELOW_LOW_THRESHOLD).sum().item()))
    df = pd.concat(rows)
    df["bucket"] = pd.cut(df["size"], [0, 32, 96, 1e9], labels=["small", "medium", "large"])
    g = df.groupby("bucket", observed=False)
    A, P, N = np.mean(per_img, axis=0)
    return {"anchors / image": A, "positives / image": P, "negatives / image": N, "negatives per positive": N / P,
            **{f"positives >= thr per GT, {k}": v for k, v in g.positives.mean().items()},
            **{f"% GT without anchor >= thr, {k}": v * 100 for k, v in g.low_quality_only.mean().items()}}

## 3. Speed test

Time per training iteration (including data loading), inference latency and GPU memory, to check that the epochs fit the budget.

In [ ]:
if RUN_SPEED_TEST and RUN & {"baseline", "anchors", "iou", "loss"}:
    seed_everything()
    m = build_retina().to(DEVICE)
    opt = torch.optim.SGD([p for p in m.parameters() if p.requires_grad], lr=CFG["lr"], momentum=0.9)
    n = 8 if QUICK else 48
    torch.cuda.reset_peak_memory_stats() if DEVICE.type == "cuda" else None
    h = fit(m, make_loader(DetDataset("train", train=True, label_offset=0, max_frames=n), CFG["batch_size"], num_workers=CFG["num_workers"]), opt, 1)
    sec_per_img = h[0]["time_s"] / n
    inf = evaluate_model(torchvision_predictor(m), "val", max_frames=20, batch_size=1, num_workers=CFG["num_workers"], label_offset=0)
    n_train = len(DetDataset("train", max_frames=CFG["train_frames"]))
    n_val = len(DetDataset("val", max_frames=CFG["val_frames"]))
    epoch_min = (sec_per_img * n_train + inf["ms_per_img"] / 1000 * 1.5 * n_val) / 60   # train + val (x1.5 for loading/COCOeval)
    print(f"train {sec_per_img:.2f} s/img | inference {inf['ms_per_img']:.0f} ms/img | "
          f"peak memory {torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == 'cuda' else 0:.1f} GB")
    print(f"estimated epoch incl. validation: {epoch_min:.0f} min -> run of {CFG['epochs']} epochs: {epoch_min * CFG['epochs']:.0f} min")
    del m, opt
    torch.cuda.empty_cache()

## 4. Baseline

Torchvision defaults: anchor base sizes 32–512 px (32–812 px with the 3 scales), IoU thresholds 0.5/0.4, GIoU box loss, NMS IoU 0.5.

In [ ]:
if "baseline" in RUN:
    train_stage("baseline", **DEFAULT_MODEL)
base = json.load(open(ensure_run("baseline") / "metrics.json"))
pd.Series({k: v for k, v in base.items() if k != "per_class"}).round(3).to_frame("baseline").T

## 5. Experiment 1 – Anchor sizes

RetinaNet has no P2 level: its finest level P3 has stride 8 and default anchors from 32 px, while 54% of the boxes are under 32 px at the input size (notebook 01, section 2.5).

Data-grounded base sizes **13/26/52/104/208**: the smallest k-means size from notebook 01 (13 px), doubled per level like the stride, so every level keeps the same anchor/stride ratio (1.6). With the 3 scales per level they cover 13–330 px (the 1st–99.9th percentile of box sizes is 9–192 px). The k-means sizes themselves (13–87 px) are not used one per level: on P6/P7 (stride 64/128) they would be smaller than the stride and leave gaps between neighbouring anchors.

Below: how well the anchor shapes fit the training boxes (best IoU of each box with any anchor shape, position ignored).

In [ ]:
pd.DataFrame({"default 32-812": anchor_fit(anchor_sizes(DEFAULT_BASES)),
              "data 13-330": anchor_fit(anchor_sizes(DATA_BASES))}).T.round(2)

In [ ]:
if "anchors" in RUN:
    train_stage("anchors", **{**DEFAULT_MODEL, "bases": DATA_BASES})
anch = json.load(open(ensure_run("anchors") / "metrics.json"))
pd.DataFrame({s: {k: v for k, v in m.items() if k != "per_class"} for s, m in [("baseline", base), ("anchors", anch)]}).T.round(3)

_TODO: findings after the run._

## 6. Experiment 2 – IoU thresholds of the anchor assignment

During training every anchor is labelled by its best IoU with a GT box: positive (learns the class and the box) if IoU ≥ the foreground threshold, background if IoU < the background threshold, ignored in between. Every GT box also keeps its best anchor as a low-quality positive. Small boxes rarely reach IoU 0.5 with any anchor, so they get few positives. Lowering the thresholds to **0.4/0.3** gives them more positives, at the cost of looser positives (possibly worse AP75). The experiment uses the better anchors from experiment 1.

Below: anchor labels on 200 train frames for each anchor set and threshold pair (no training needed).

In [ ]:
stats = {f"{a} anchors, IoU {t[0]}/{t[1]}": assignment_stats(b, t)
         for a, b in [("default", DEFAULT_BASES), ("data", DATA_BASES)] for t in [DEFAULT_IOU, LOW_IOU]}
pd.DataFrame(stats).T.round(1)

In [ ]:
if "iou" in RUN:
    train_stage("iou", **{**model_cfg(best_so_far(["baseline", "anchors"])), "iou": LOW_IOU})
iou = json.load(open(ensure_run("iou") / "metrics.json"))
pd.DataFrame({s: {k: v for k, v in m.items() if k != "per_class"}
              for s, m in [("baseline", base), ("anchors", anch), ("iou", iou)]}).T.round(3)

_TODO: findings after the run._

## 7. Experiment 3 (optional) – Box regression loss

Loss modification on the best configuration so far: the GIoU loss of torchvision's v2 recipe vs smooth L1 on the box offsets (the v1 recipe and the loss Faster R-CNN uses in notebook 02). GIoU optimises the overlap directly and is scale-invariant; smooth L1 weights errors by the offset encoding. Run only if the time budget allows.

In [ ]:
if "loss" in RUN:
    train_stage("loss", **{**model_cfg(best_so_far(["baseline", "anchors", "iou"])), "box_loss": "smooth_l1"})
try:
    loss = json.load(open(ensure_run("loss") / "metrics.json"))
    display(pd.Series({k: v for k, v in loss.items() if k != "per_class"}).round(3).to_frame("loss").T)
except FileNotFoundError:
    print("optional stage 'loss' was not run")

## 8. Tuned model and comparison with Faster R-CNN

Final model ("tuned"): the best training by val mAP with the better final NMS IoU (0.5 or 0.6, test time). Its val predictions are saved for the analysis below. Compared with the tuned Faster R-CNN from notebook 02 on the same val frames, with the same epochs and about the same training time. Table rows: anchors, IoU thresholds and box loss used in training, then the NMS IoU.

In [ ]:
TRAINED = ["baseline", "anchors", "iou", "loss"]
if "final" in RUN:
    src = best_so_far(TRAINED)
    model = load_model(src)
    ds = DetDataset("val", label_offset=0, max_frames=CFG["val_frames"])
    rows, best = [], None
    for t in NMS:
        p, ms = predict_dataset(torchvision_predictor(set_test_params(model, t)), ds, batch_size=1, num_workers=CFG["num_workers"])
        m = {**coco_evaluate(p, "val", image_ids=ds.image_ids), "ms_per_img": ms}
        rows.append({"nms": t, **{k: v for k, v in m.items() if k != "per_class"}})
        if best is None or m["mAP"] > best[1]["mAP"]:
            best = (t, m, p)
    t, metrics, p = best
    d = save_run(name("final"), config={"trained": name(src), "model": model_cfg(src), "test": {"nms": t}, "image_ids": ds.image_ids},
                 metrics=metrics, preds=p)
    pd.DataFrame(rows).to_csv(d / "nms.csv", index=False)
    if not QUICK:
        push_run(name("final"))
    del model
d = ensure_run("final")
final_cfg, final = json.load(open(d / "config.json")), json.load(open(d / "metrics.json"))
preds = json.load(open(d / "preds_val.json"))
pd.read_csv(d / "nms.csv").set_index("nms").round(3)

In [ ]:
# Tuned two-stage model from notebook 02 (metrics, predictions and training history, no weights)
for r in ["frcnn_final", "frcnn_baseline"]:
    if not (RUNS_DIR / r / "metrics.json").exists():
        pull_run(r, weights=False)
frcnn_cfg = json.load(open(RUNS_DIR / "frcnn_final" / "config.json"))
frcnn = json.load(open(RUNS_DIR / "frcnn_final" / "metrics.json"))
frcnn_preds = json.load(open(RUNS_DIR / "frcnn_final" / "preds_val.json"))


def row_name(stage):
    # anchors, IoU thresholds and box loss used in training + NMS IoU at test time
    c = json.load(open(ensure_run(stage) / "config.json"))
    mc = c["model"]
    anchors = "data" if list(mc["bases"]) == list(DATA_BASES) else "default"
    return f"{'final: ' if stage == 'final' else ''}{anchors} anchors, IoU {mc['iou'][0]}/{mc['iou'][1]}, {mc['box_loss']}, NMS {c['test']['nms']}"


stages = [s for s in TRAINED + ["final"] if (RUNS_DIR / name(s) / "metrics.json").exists()]
FINAL_NAME = row_name("final")
cols = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR", "ms_per_img"]
res = load_results()
table = res.reindex(["frcnn_final"] + [name(s) for s in stages])[cols]
table.index = ["Faster R-CNN, tuned (notebook 02)"] + [row_name(s) for s in stages]
table

In [ ]:
# Training curves (same budget as Faster R-CNN) and per-class AP
plot_history({**{s: json.load(open(ensure_run(s) / "history.json")) for s in stages if s != "final"},
              "Faster R-CNN baseline": json.load(open(RUNS_DIR / "frcnn_baseline" / "history.json"))})
pd.DataFrame({"Faster R-CNN, tuned": frcnn["per_class"], FINAL_NAME: final["per_class"]}).T.round(3)

_TODO: findings after the run._

## 9. What removing the proposal stage costs

- **Foreground/background imbalance.** Without the RPN, the classifier scores every anchor (about 170k per frame, table in section 6) against ~46 objects, so almost all anchors are easy background, and with cross-entropy their summed loss dominates. RetinaNet addresses it with **focal loss** (γ = 2 down-weights well-classified anchors) and the **prior-probability bias init** (0.01), so training does not start with a flood of background loss. Faster R-CNN instead keeps 1000 proposals and samples 512 RoIs per image with up to 25% positives.
- **No second look.** Faster R-CNN pools features for each proposal (RoIAlign) and classifies and refines it again; RetinaNet predicts class and box once, from a fixed grid location. Fine class decisions (car vs van) and localisation rely on one dense head. Addressed by IoU-based box losses (GIoU), quality-aware scores (IoU/centerness branches, GFL) or cascade refinement.
- **Small objects depend only on the anchor assignment.** There is no proposal stage to recover them, and RetinaNet has no stride-4 level. A small box that no anchor matches well gets few positives (section 6). Addressed by data-grounded anchors and lower IoU thresholds (experiments 1–2), or adaptive assignment (ATSS).
- **Speed.** No per-proposal computation, but dense heads on P3 at 1280 px are not free (ms per image in section 8).

Measured below: share of GT boxes covered by a detection (IoU ≥ 0.5, any class, score ≥ 0.3) by size, and detections per image, for both tuned models.

In [ ]:
gt, det = match_errors(preds, final_cfg["image_ids"])
gt2, det2 = match_errors(frcnn_preds, frcnn_cfg["image_ids"])


def covered(g):
    f = g.outcome.isin(["correct", "crowded", "wrong_class"])
    r = f.groupby(g.bucket.astype(str)).mean()
    r["all"] = f.mean()
    return r


cmp = pd.DataFrame({"Faster R-CNN, tuned": covered(gt2), "RetinaNet, tuned": covered(gt)}).reindex(["small", "medium", "large", "all"])
cmp.loc["detections / image (score >= 0.3)"] = [len(det2) / len(frcnn_cfg["image_ids"]), len(det) / len(final_cfg["image_ids"])]
cmp.round(3)

_TODO: findings after the run._

## 10. Failure cases

Errors of the tuned model at score ≥ 0.3 and IoU 0.5: outcome of every GT box by size and occlusion, class confusions and the types of false positives (same definitions as in notebook 02).

GT outcomes: **correct** = detected with the right class; **crowded** = covered only by a box already used for a neighbour; **wrong_class** = detected as another class; **mislocalized** = best IoU 0.1–0.5; **missed** = no detection. Detections: **tp** = matches a GT box; **background** = on nothing; **ignored** = centre in an ignore region; **wrong_class** = right place, wrong class; **duplicate** = second box on an already matched GT.

In [ ]:
error_tables(gt, det)
plot_confusion(gt);

Frames with the most missed small objects, the most class confusions and the most background false positives. Top: ground truth (box colour = class). Bottom: model predictions with score ≥ 0.3 (box colour = predicted class). Red areas are ignore regions.

In [ ]:
plot_failure_frames(gt, det, preds);

_TODO: failure cases and their likely causes after the run._

## 11. Conclusion

_TODO: after the run._